# Módulo 1 · 03 — Modelos de referencia

- **Ingenuo estacional**: repite la última semana observada; P90 = pronóstico + cuantil 0.9 de sus errores históricos.
- **Holt-Winters amortiguado** sobre `log1p(y)`, estacionalidad semanal; P90 por residuos.

Evaluación con origen rodante: cada `paso_origen` días en el periodo de prueba se pronostican 30 días usando sólo la historia hasta el origen.

In [ ]:
# parameters
DEMO = False

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "modulo1_demanda"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common.seeds import fijar_semilla, semillas_experimento
from common.config import cargar_config
import m1lib as m1

fijar_semilla()
cfg = cargar_config("modulo1")
M1 = RAIZ / "modulo1_demanda"
DATA, FIG, MOD = M1 / "data", M1 / "figures", M1 / "models"
H, TAU = cfg["pronostico"]["horizonte"], cfg["pronostico"]["cuantil_alto"]
plt.rcParams["figure.figsize"] = (12, 4)

Y = pd.read_parquet(DATA / "processed" / "series_diarias.parquet")       # pasajeros/ruta/día (imputada)
cal = pd.read_parquet(DATA / "processed" / "calendario.parquet")
rutas = pd.read_csv(DATA / "processed" / "rutas.csv", dtype=str)
INI_VAL, INI_TEST = pd.Timestamp(cfg["datos"]["inicio_validacion"]), pd.Timestamp(cfg["datos"]["inicio_prueba"])
RUTAS_IDA = rutas.loc[rutas.sentido == "ida", "ruta"].tolist()
if DEMO:
    RUTAS_IDA = RUTAS_IDA[:cfg["demo"]["rutas"]]
Y_ida = Y[RUTAS_IDA]
origenes_test = m1.origenes_rodantes(Y.index, INI_TEST, H, cfg["pronostico"]["paso_origen"])
escalas_mase = {r: m1.escala_mase(Y_ida.loc[:INI_VAL - pd.Timedelta(days=1), r].dropna().values) for r in RUTAS_IDA}
print(f"{len(RUTAS_IDA)} rutas · {len(origenes_test)} orígenes de prueba: {origenes_test[0].date()} … {origenes_test[-1].date()}")

In [ ]:
import time, warnings
warnings.filterwarnings("ignore")
registros, tiempos = [], {}
for nombre_mod, fn in {"ingenuo_estacional": m1.ingenuo_estacional, "holt_winters": m1.ets_semanal}.items():
    t0 = time.time()
    for t in origenes_test:
        fechas = pd.date_range(t + pd.Timedelta(days=1), periods=H)
        for r in RUTAS_IDA:
            pred, q = fn(Y_ida.loc[:t, r].dropna(), H, TAU)
            registros.append(pd.DataFrame({"modelo": nombre_mod, "ruta": r, "origen": t, "fecha": fechas,
                                           "h": np.arange(1, H + 1), "y": Y_ida.loc[fechas, r].values,
                                           "pred": pred, "q_alto": q}))
    tiempos[nombre_mod] = time.time() - t0
reg = pd.concat(registros, ignore_index=True)
reg.to_parquet(DATA / "processed" / "pred_baselines.parquet")
print({k: f"{v:.1f}s" for k, v in tiempos.items()})

In [ ]:
tab = m1.tabla_errores(reg, escalas_mase, TAU)
display(tab.groupby("modelo")[["MAE", "RMSE", "MASE", f"cobertura_P{int(TAU*100)}", "pinball"]].mean().round(3))
tab.round(3)

In [ ]:
m1.errores_por_bloque(reg).round(1)

In [ ]:
# Error por horizonte (1..30) — se completa con las redes en el notebook 04
eh = reg.assign(ae=(reg.y - reg.pred).abs()).groupby(["modelo", "h"]).ae.mean().unstack(0)
eh.plot(marker=".", title="MAE por horizonte — modelos de referencia"); plt.xlabel("h (días)")

Parámetros entrenables: ingenuo = 0; Holt-Winters = nivel, tendencia, amortiguamiento + 7 estacionales + 3 suavizamientos ≈ 13 por ruta (verificar con `fit.params`). Presupuesto de búsqueda: 1 configuración por familia.